# 04-01 Backtest

Backtests the lead-lag strategy on every deduplicated correlation pair: when the leader gains >= threshold on a trading day, buy the follower and hold for `lag_days` trading days. Trades are joined to a market day-over-day benchmark and uploaded to `backtest/<run>/data.parquet`.

In [ ]:
# ============================================================================
# SETUP -- installs, imports, config (env vars / config.json -- never hardcoded)
# ============================================================================

# --- Install packages (no-op if already present) --------------------------
# !pip install -q duckdb --upgrade
import os
import json
import sys
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
import duckdb
# --- Configuration --------------------------------------------------------
# Secrets resolve in priority order:
#   1. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
#      AWS_REGION, S3_BUCKET, MASSIVE_API_KEY, ...)
#   2. config.json in the current directory (see config.example.json)
#   3. Built-in defaults (non-secret values only)
# On Kaggle: set secrets via notebook settings (Add-ons -> Secrets), which
# are injected as environment variables.
CONFIG_FILE = "config.json"


def get_secret(name, default=""):
    val = os.environ.get(name)
    if val:
        return val
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            if name in data:
                return str(data[name])
        except (OSError, ValueError):
            pass
    return default


class Config:
    def __init__(self):
        self.aws_access_key_id = ""
        self.aws_secret_access_key = ""
        self.aws_region = "us-east-1"
        self.s3_bucket = "market-data-zw"
        self.massive_api_key = ""

        # Paths (S3 keys under the bucket)
        self.types_prefix = "parquet_data/types"
        self.tickers_prefix = "parquet_data/summary/tickers"
        self.ticker_details_prefix = "parquet_data/summary/ticker_yahoo_details"
        self.minute_staging_prefix = "parquet_data/minute_data_staging"
        self.minute_final_prefix = "parquet_data/minute_data_final"
        self.minute_summary_prefix = "parquet_data/summary/minute_summary"
        self.daily_volume_prefix = "parquet_data/summary/daily_volume"
        self.correlation_prefix = "parquet_data/strategies/correlation"
        self.backtest_prefix = "parquet_data/backtest"
        self.backtest_metrics_prefix = "parquet_data/analysis/backtest_metrics"

        # Spark
        self.spark_executor_memory = "24g"
        self.spark_executor_cores = 4
        self.spark_driver_memory = "24g"
        self.spark_tmp = "/tmp/spark"


def load_config():
    cfg = Config()
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            for key, value in data.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, value)
        except (OSError, ValueError) as e:
            print(f"[config] WARNING: could not load {CONFIG_FILE}: {e}")

    env_map = {
        "AWS_ACCESS_KEY_ID": "aws_access_key_id",
        "AWS_SECRET_ACCESS_KEY": "aws_secret_access_key",
        "AWS_REGION": "aws_region",
        "S3_BUCKET": "s3_bucket",
        "MASSIVE_API_KEY": "massive_api_key",
        "SPARK_DRIVER_MEMORY": "spark_driver_memory",
        "SPARK_EXECUTOR_MEMORY": "spark_executor_memory",
        "SPARK_EXECUTOR_CORES": "spark_executor_cores",
    }
    for env_name, attr in env_map.items():
        val = os.environ.get(env_name)
        if val:
            if attr == "spark_executor_cores":
                val = int(val)
            setattr(cfg, attr, val)
    return cfg
# --- S3 helpers -----------------------------------------------------------
def s3_client(cfg):
    from botocore.config import Config as BotocoreConfig
    config = BotocoreConfig(retries={"max_attempts": 5, "mode": "adaptive"},
                            connect_timeout=30, read_timeout=60)
    return boto3.client("s3",
                        aws_access_key_id=cfg.aws_access_key_id,
                        aws_secret_access_key=cfg.aws_secret_access_key,
                        region_name=cfg.aws_region,
                        config=config)


def upload_parquet(df, s3, bucket, key, compression="snappy"):
    buf = BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression=compression,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())


def download_parquet(s3, bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))


def list_s3_keys(s3, bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def tickers_from_prefix(s3, bucket, prefix):
    """Ticker symbols from `<prefix>/<TICKER>.parquet` object keys."""
    return [k.split("/")[-1][:-len(".parquet")] for k in list_s3_keys(s3, bucket, prefix)
            if k.endswith(".parquet")]


def duckdb_s3_connect(cfg):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET s3_access_key_id='{cfg.aws_access_key_id}';")
    con.execute(f"SET s3_secret_access_key='{cfg.aws_secret_access_key}';")
    con.execute(f"SET s3_region='{cfg.aws_region}';")
    return con


def spark_session(cfg):
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MarketDataPlatform")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
        .config("spark.executor.memory", cfg.spark_executor_memory)
        .config("spark.executor.cores", str(cfg.spark_executor_cores))
        .config("spark.driver.memory", cfg.spark_driver_memory)
        .config("spark.hadoop.fs.s3a.access.key", cfg.aws_access_key_id)
        .config("spark.hadoop.fs.s3a.secret.key", cfg.aws_secret_access_key)
        .config("spark.hadoop.fs.s3a.endpoint", f"s3.{cfg.aws_region}.amazonaws.com")
        .config("spark.local.dir", cfg.spark_tmp)
        .config("spark.hadoop.tmp.dir", cfg.spark_tmp)
        .config("spark.sql.warehouse.dir", f"{cfg.spark_tmp}/warehouse")
        .getOrCreate()
    )
    spark.conf.set("spark.hadoop.fs.s3a.committer.name", "directory")
    spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    spark.conf.set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "append")
    spark.conf.set("spark.sql.debug.maxToStringFields", "100")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.ansi.enabled", "false")
    spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
    spark.conf.set("spark.sql.parquet.mergeSchema", "true")
    spark.sparkContext.setLogLevel("ERROR")
    return spark

os.makedirs("/tmp/spark", exist_ok=True)

# --- Instantiate config + clients -----------------------------
cfg = load_config()
s3 = s3_client(cfg)
spark = spark_session(cfg)
print("Setup complete")
print(f"Bucket: {cfg.s3_bucket} | Region: {cfg.aws_region}")


In [ ]:
# ============================================================================
# Data source resolution -- prefer the local Kaggle dataset mirror (created
# by 02-02-s3-to-kaggle-dataset: fast, free reads), fall back to S3.
# ============================================================================

import glob as _glob

MIRROR_CANDIDATES = [
    "/kaggle/input/datasets/dsptlp/market-data-s3-dataset/s3_data/parquet_data",
    "/kaggle/input/market-data-s3-dataset/s3_data/parquet_data",
]


def resolve(rel, name="", use_glob=False):
    """Kaggle-mirror path when mounted, else s3a:// URI."""
    short = rel[len("parquet_data/"):] if rel.startswith("parquet_data/") else rel
    for root in MIRROR_CANDIDATES:
        local = os.path.join(root, short, name)
        if use_glob:
            if _glob.glob(local):
                return local
        elif os.path.exists(local):
            return local
    return f"s3a://{cfg.s3_bucket}/{rel}/{name}"


In [ ]:
# ============================================================================
# Backtest engine -- self-contained helpers (no external package imports)
# ============================================================================

def combine_trades(df_backtest_results: pd.DataFrame) -> pd.DataFrame:
    """Concatenate all per-pair trade DataFrames into one."""
    trades_frames = [r for r in df_backtest_results["trades"].tolist() if r is not None]
    if not trades_frames:
        return pd.DataFrame()
    return pd.concat(trades_frames, ignore_index=True)

def join_market_dod(trades_df: pd.DataFrame, market_dod: pd.DataFrame) -> pd.DataFrame:
    """Join market day-over-day returns onto backtest trades by signal date."""
    trades_df = trades_df.copy()
    trades_df["signal_date"] = pd.to_datetime(trades_df["signal_date"])
    return pd.merge(trades_df, market_dod, left_on="signal_date", right_on="date", how="left")


from __future__ import annotations

from datetime import datetime
from io import BytesIO
from typing import Any

import pandas as pd
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F


# =============================================================================
# Market benchmark
# =============================================================================

def calc_market_dod(
    df: DataFrame,
    start_date: str | None = None,
    end_date: str | None = None,
    outlier_bound: float = 50,
) -> pd.DataFrame:
    """
    Day-over-day market return across all tickers (median to resist outliers).

    Returns a pandas DataFrame with columns:
    ``date``, ``market_dod_pct`` (median), ``market_dod_mean``, ``tickers_counted``.
    """
    filtered = df
    if start_date:
        filtered = filtered.filter(F.col("date") >= start_date)
    if end_date:
        filtered = filtered.filter(F.col("date") <= end_date)

    ticker_window = Window.partitionBy("ticker").orderBy("date")
    dod = (
        filtered.select("ticker", "date", "price")
        .withColumn("prev_price", F.lag("price").over(ticker_window))
        .withColumn(
            "dod_pct",
            ((F.col("price") - F.col("prev_price")) / F.col("prev_price")) * 100,
        )
        .filter(F.col("dod_pct").isNotNull())
        .filter(F.col("dod_pct").between(-outlier_bound, outlier_bound))
    )
    market_returns = (
        dod.groupBy("date")
        .agg(
            F.percentile_approx("dod_pct", 0.5).alias("market_dod_pct"),
            F.avg("dod_pct").alias("market_dod_mean"),
            F.count("ticker").alias("tickers_counted"),
        )
        .orderBy("date")
        .toPandas()
    )
    return market_returns


# =============================================================================
# Single-pair backtest
# =============================================================================

def backtest_strategy(
    df: DataFrame,
    leader: str,
    follower: str,
    start_date: str,
    end_date: str,
    leader_threshold: float = 0.03,
    holding_days: int = 14,
    shares_per_trade: int = 100,
    initial_capital: float = 10000.0,
    correlation: float = 0.0,
    commission_pct: float = 0.001,
) -> dict[str, Any] | None:
    """
    Backtest the lead-lag strategy for a single pair.

    Parameters
    ----------
    df : DataFrame
        Daily prices with columns ``ticker``, ``date``, ``price``.
    leader, follower : str
        Leader (trigger) and follower (instrument) tickers.
    start_date, end_date : str
        Inclusive backtest window ``YYYY-MM-DD``.
    leader_threshold : float
        Minimum leader daily gain to trigger a buy (e.g. ``0.02`` = 2%).
    holding_days : int
        Trading days to hold the follower.
    shares_per_trade : int
        Shares per trade.
    initial_capital : float
        Starting capital for the running capital curve.
    correlation : float
        Correlation recorded with the pair (carried into the trades).
    commission_pct : float
        Commission per trade side, fraction of trade value.

    Returns
    -------
    ``{'trades': DataFrame, 'summary': dict}`` or ``None`` if no trades.
    """
    leader_data = (
        df.filter(
            (F.col("ticker") == leader)
            & (F.col("date") >= start_date)
            & (F.col("date") <= end_date)
        )
        .select("date", "price")
        .orderBy("date")
        .withColumn("prev_price", F.lag("price").over(Window.orderBy("date")))
        .withColumn("return", (F.col("price") - F.col("prev_price")) / F.col("prev_price"))
        .filter(F.col("return").isNotNull())
        .toPandas()
    )

    signals = leader_data[leader_data["return"] >= leader_threshold].copy()
    if len(signals) == 0:
        return None

    follower_data = (
        df.filter(
            (F.col("ticker") == follower) & (F.col("date") >= start_date)
        )
        .select("date", "price")
        .orderBy("date")
        .toPandas()
    )
    follower_prices = dict(zip(follower_data["date"], follower_data["price"], strict=True))

    trades: list[dict[str, Any]] = []
    capital = initial_capital

    for _, signal in signals.iterrows():
        buy_date = signal["date"]
        try:
            buy_idx = follower_data.index[follower_data["date"] == buy_date][0]
        except IndexError:
            continue
        sell_idx = min(buy_idx + holding_days, len(follower_data) - 1)
        sell_date = follower_data.iloc[sell_idx]["date"]

        buy_price = follower_prices.get(buy_date)
        sell_price = follower_prices.get(sell_date)
        if buy_price is None or sell_price is None:
            continue

        cost = buy_price * shares_per_trade * (1 + commission_pct)
        revenue = sell_price * shares_per_trade * (1 - commission_pct)
        profit = revenue - cost
        commission = (
            buy_price * shares_per_trade * commission_pct
            + sell_price * shares_per_trade * commission_pct
        )
        profit_pct = (profit / cost) * 100
        capital += profit

        trades.append({
            "trade_num": len(trades) + 1,
            "signal_date": signal["date"],
            "leader_gain": signal["return"] * 100,
            "buy_date": buy_date,
            "sell_date": sell_date,
            "buy_price": buy_price,
            "sell_price": sell_price,
            "shares": shares_per_trade,
            "profit": profit,
            "profit_pct": profit_pct,
            "commission": commission,
            "commission_pct": commission_pct,
            "capital": capital,
            "leader": leader,
            "follower": follower,
            "holding_days": holding_days,
            "leader_threshold": leader_threshold,
            "start_date": start_date,
            "end_date": end_date,
            "correlation": correlation,
        })

    if not trades:
        return None

    trades_df = pd.DataFrame(trades)
    total_profit = trades_df["profit"].sum()
    total_return = ((capital - initial_capital) / initial_capital) * 100
    wins = int((trades_df["profit"] > 0).sum())
    losses = int((trades_df["profit"] < 0).sum())

    summary = {
        "signals": len(signals),
        "trades": len(trades_df),
        "initial_capital": initial_capital,
        "final_capital": capital,
        "total_profit": total_profit,
        "total_return_pct": total_return,
        "total_commission": trades_df["commission"].sum(),
        "commission_pct": commission_pct,
        "wins": wins,
        "losses": losses,
        "win_rate": (wins / len(trades_df)) * 100,
        "avg_profit": trades_df["profit"].mean(),
        "best_trade": trades_df["profit"].max(),
        "worst_trade": trades_df["profit"].min(),
    }
    return {"trades": trades_df, "summary": summary}


# =============================================================================
# Batch backtest over correlation pairs
# =============================================================================

def run_backtest_on_all_correlations(
    df_correlations: pd.DataFrame,
    price_df: DataFrame,
    threshold: float = 0.02,
    commission_pct: float = 0.001,
    start_date: str = "2026-01-01",
    end_date: str = "2026-08-31",
    ticker1_col: str = "leader",
    ticker2_col: str = "follower",
    correlation_col: str = "correlation",
    holding_days_col: str = "lag_days",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Run :func:`backtest_strategy` over every pair and collect the results.

    Returns a DataFrame with one row per pair containing ``trades``,
    ``summary``, ``original_correlation`` and ``pair_id``.
    """
    from tqdm import tqdm

    backtest_results = []
    rows = df_correlations.to_dict("records")

    for idx, row in enumerate(tqdm(rows, desc="Running backtests", disable=not verbose)):
        ticker1 = row[ticker1_col]
        ticker2 = row[ticker2_col]
        holding_days = int(row.get(holding_days_col, 14))
        correlation = row.get(correlation_col, 0)

        result = backtest_strategy(
            price_df,
            leader=ticker1,
            follower=ticker2,
            holding_days=holding_days,
            leader_threshold=threshold,
            start_date=start_date,
            end_date=end_date,
            correlation=correlation,
            commission_pct=commission_pct,
        )
        if result is None or not isinstance(result, dict):
            continue

        result["original_correlation"] = row.get(correlation_col, None)
        result["pair_id"] = idx
        backtest_results.append(result)

        if verbose and (idx + 1) % 10 == 0:
            successful = sum(
                1 for r in backtest_results if r.get("summary", {}).get("trades", 0) > 0
            )
            print(f"   Processed {idx + 1}/{len(rows)} pairs. Successful: {successful}")

    if backtest_results:
        return pd.DataFrame(backtest_results)
    return pd.DataFrame(columns=["trades"])


def upload_backtest_results(
    trades_df: pd.DataFrame,
    s3: Any,
    bucket: str,
    run_timestamp: str | None = None,
) -> str:
    """Upload backtest trades to S3 as Parquet. Returns the S3 key."""
    if run_timestamp is None:
        run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    trades_df = trades_df.copy()
    trades_df["run_timestamp"] = run_timestamp
    for col in trades_df.columns:
        if pd.api.types.is_datetime64_any_dtype(trades_df[col]):
            trades_df[col] = trades_df[col].astype(str)

    file_key = f"parquet_data/backtest/{run_timestamp}/data.parquet"
    buffer = BytesIO()
    trades_df.to_parquet(
        buffer,
        index=False,
        engine="pyarrow",
        coerce_timestamps="ms",
        allow_truncated_timestamps=True,
    )
    buffer.seek(0)
    s3.put_object(Bucket=bucket, Key=file_key, Body=buffer.getvalue())
    return file_key


# =============================================================================
# Pipeline (notebook 04-01)
# =============================================================================

def load_daily_prices(spark, minute_summary_path: str) -> DataFrame:
    """Daily close prices (last minute bar of each day) with trade counts."""
    df_raw = spark.read.parquet(minute_summary_path).filter(F.col("rn_desc").isin(1))
    return df_raw.select(
        F.col("symbol").alias("ticker"),
        F.col("close").alias("price"),
        F.col("trade_date").alias("date"),
        F.col("trades").alias("volume"),
    )








In [ ]:
# ============================================================================
# All parameters (single source of truth for this notebook)
# ============================================================================

DATA_SOURCE_LOCAL = resolve(cfg.minute_summary_prefix, "data.parquet")
DATA_SOURCE_CORR  = resolve(cfg.correlation_prefix, "*/*.parquet", use_glob=True)

# --- Market DoD ---------------------------------------------------------------
MARKET_DOD_START  = "2026-01-01"
MARKET_DOD_END    = "2026-08-31"
DOD_OUTLIER_BOUND = 50        # drop |dod_pct| above this

# --- Backtest strategy --------------------------------------------------------
LEADER_THRESHOLD  = 0.03      # leader daily gain trigger
HOLDING_DAYS      = 14        # fallback when lag_days missing
SHARES_PER_TRADE  = 100
INITIAL_CAPITAL   = 10000
COMMISSION_PCT    = 0.001

# --- Correlation-pair backtest loop -------------------------------------------
THRESHOLD           = 0.02
BACKTEST_START_DATE = "2026-01-01"
BACKTEST_END_DATE   = "2026-08-31"

# --- Dedup -----------------------------------------------------------------
CORRELATION_ROUND_DIGITS = 6

spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.mergeSchema", "false")

In [ ]:
# ============================================================================
# Load daily close prices (last minute bar of each day)
# ============================================================================

df = load_daily_prices(spark, DATA_SOURCE_LOCAL)
print(f"{df.count():,} daily observations")

In [ ]:
# ============================================================================
# Day-over-day market return across all stocks (median, outlier-filtered)
# ============================================================================

market_dod = calc_market_dod(df, start_date=MARKET_DOD_START, end_date=MARKET_DOD_END,
                             outlier_bound=DOD_OUTLIER_BOUND)
print(f"Computed market returns for {len(market_dod):,} trading days")
market_dod.head(10)

In [ ]:
# ============================================================================
# Load correlation pairs (explicit schema -- runs have heterogeneous schemas)
# ============================================================================

CORRELATION_SCHEMA = (
    "leader STRING, follower STRING, correlation DOUBLE, sign_fraction DOUBLE, "
    "avg_follower_7d_price_change DOUBLE, avg_pre_7d_return DOUBLE, "
    "avg_follower_30d_price_change DOUBLE, avg_pre_30d_return DOUBLE, "
    "avg_follower_3d_price_change DOUBLE, avg_pre_3d_return DOUBLE, "
    "significance_score DOUBLE, abs_correlation DOUBLE, abs_expected_move DOUBLE, "
    "num_observations BIGINT, recent_30d_correlation DOUBLE, "
    "recent_10d_correlation DOUBLE, recent_3d_correlation DOUBLE, "
    "last_observation_date STRING, rn INT, run_timestamp STRING, "
    "iteration BIGINT, lag_days BIGINT, lookback_days BIGINT, "
    "persistence_window BIGINT"
)

df_correlations_raw = spark.read.schema(CORRELATION_SCHEMA).parquet(DATA_SOURCE_CORR)

# Round correlation before deduplication
df_clean = (
    df_correlations_raw
    .withColumn("correlation_rounded", F.round(F.col("correlation"), CORRELATION_ROUND_DIGITS))
    .dropDuplicates(["leader", "follower", "correlation_rounded",
                     "lag_days", "lookback_days", "persistence_window"])
    .drop("correlation_rounded")
)
df_correlations_pd = df_clean.toPandas()
print(f"{len(df_correlations_pd):,} unique correlation pairs")

In [ ]:
# ============================================================================
# Run backtest_strategy on ALL correlation pairs (collects results)
# ============================================================================

from tqdm.notebook import tqdm

backtest_results = []

for idx, row in tqdm(df_correlations_pd.iterrows(), total=len(df_correlations_pd),
                     desc="Running backtests"):
    ticker1 = row["leader"]
    ticker2 = row["follower"]
    holding_days = int(row.get("lag_days", HOLDING_DAYS))
    correlation = row["correlation"]

    result = backtest_strategy(
        df, leader=ticker1, follower=ticker2,
        holding_days=holding_days, leader_threshold=THRESHOLD,
        start_date=BACKTEST_START_DATE, end_date=BACKTEST_END_DATE,
        correlation=correlation, commission_pct=COMMISSION_PCT,
    )
    if result is None or not isinstance(result, dict):
        continue
    result["original_correlation"] = row.get("correlation", None)
    result["pair_id"] = idx
    backtest_results.append(result)

print(f"\nBacktest complete: {len(backtest_results)} pairs produced trades")

In [ ]:
# ============================================================================
# Combine all trades, join market DoD, upload to S3
# ============================================================================

from datetime import datetime

df_backtest_results = pd.DataFrame(backtest_results)
trades_df = combine_trades(df_backtest_results)
trades_df = join_market_dod(trades_df, market_dod)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")
trades_df["run_timestamp"] = run_timestamp

for col in trades_df.columns:
    if pd.api.types.is_datetime64_any_dtype(trades_df[col]):
        trades_df[col] = trades_df[col].astype(str)

file_key = f"{cfg.backtest_prefix}/{run_timestamp}/data.parquet"
upload_parquet(trades_df, s3, cfg.s3_bucket, file_key)
print(f"Uploaded {len(trades_df):,} trades -> s3://{cfg.s3_bucket}/{file_key}")